In [1]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [2]:
import sys

In [3]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers.table import TableChunker
from financial_qa.chunkers.table_aware_recursive import TableAwareRecursiveChunker
from financial_qa.chunkers.synthetic_summary import SummaryChunker
from financial_qa.chunkers.table_split import TableSplitChunker
from financial_qa.chunkers.semantic import SemanticChunker
from financial_qa.chunkers.synthetic_table_row_to_text import TableRowToTextChunker
from financial_qa.preprocessors import RegulatoryReportPreprocessor
from financial_qa.embedders import GigaEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.agent.gigachat_agent_loop import GigaChatAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [20]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
if not GIGACHAT_CREDENTIALS:
    raise ValueError('GIGACHAT_CREDENTIALS is required')

DATASET_FILE = 'dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = None

RAG_DB = 'giga_embeddings_summary_table_split_semantic'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
SYNTH_CHUNK_MODEL = "google/gemma-4-26b-a4b-it"
EMBED_MODEL = 'EmbeddingsGigaR'
GIGACHAT_SCOPE = 'GIGACHAT_API_PERS'

GEN_MODEL = "GigaChat-2-Pro"
QUERY_CONCURRENCY = 10

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
# JUDGE_MODEL = 'google/gemma-4-26b-a4b-it'
JUDGE_PROCESSES = 100  # None => one process per question
USE_GIGACHAT_JUDGE = True

In [5]:
all_records = load_jsonl(DATASET_FILE)

In [6]:
len(all_records)

449

In [7]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
    else:
        print('huy')
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

449
Loaded 449 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


In [8]:
RAW_DATA_DIR = 'data/parsed'
DATA_DIR = 'data/preprocessed'

preprocessed_dir = Path(DATA_DIR)
has_preprocessed = preprocessed_dir.exists() and any(preprocessed_dir.rglob('*.md'))
if not has_preprocessed:
    print(f'Preprocessing {RAW_DATA_DIR} -> {DATA_DIR}...')
    preprocessor = RegulatoryReportPreprocessor()
    outputs = preprocessor.preprocess_dir(RAW_DATA_DIR, DATA_DIR)
    print(f'Preprocessed {len(outputs)} file(s).')
else:
    print(f'Using existing preprocessed data at {preprocessed_dir}')

Using existing preprocessed data at preprocessed_data


In [9]:
chunker = SummaryChunker(
    base_chunker=TableSplitChunker(
            text_chunker=SemanticChunker(
                chunk_size=CHUNK_SIZE,
                chunk_overlap=CHUNK_OVERLAP,
            ),
            max_rows_per_chunk=5,
        ),
    api_key=OPENROUTER_API_KEY,
    model=SYNTH_CHUNK_MODEL,
    window_size=20,
)
embedder = GigaEmbedder(
    credentials=GIGACHAT_CREDENTIALS
)

rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/preprocessed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc(max_workers=1)
else:
    print(f'Using existing index at {store_dir}')

loop = OpenRouterAgentLoop(
    rag=rag,
    max_turns=8,
    model="google/gemma-4-26b-a4b-it"
)

Using existing index at indexes/giga_embeddings_summary_table_split_semantic


In [10]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/449 [00:00<?, ?question/s]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent:   1%|          | 3/449 [00:19<33:33,  4.51s/question, avg_s=18.49, errors=0, last_conf=0.84]   

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 13s…


Querying agent:   1%|          | 5/449 [00:27<32:45,  4.43s/question, avg_s=17.69, errors=0, last_conf=0.91] 

...

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent:  98%|█████████▊| 440/449 [10:07<00:08,  1.03question/s, avg_s=13.51, errors=7, last_conf=None]

[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 16s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent: 100%|██████████| 449/449 [10:28<00:00,  1.40s/question, avg_s=13.70, errors=7, last_conf=0]   

Done: 449 answers, 7 errors


In [11]:
query_errors

[{'question_id': 'q_00d660efcf3e4607',
  'question': 'Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?',
  'answer': '',
  'evidence': [],
  'confidence': None,
  'error': 'Server disconnected',
  'elapsed_s': 72.36501904181205},
 {'question_id': 'q_b8a1328284e4a199',
  'question': 'На какой странице промежуточной раскрываемой консолидированной финансовой информации Россельхозбанка за 6 месяцев 2023 года начинается Примечание 2 «Важные оценки и профессиональные суждения в применении учётной политики»?',
  'answer': '',
  'evidence': [],
  'confidence': None,
  'error': "400, message='Bad Request', url='https://openrouter.ai/api/v1/chat/completions'",
  'elapsed_s': 4.858098458033055},
 {'question_id': 'q_14f074bad2e8efb8',
  'question': 'По какому адресу электронной почты можно направить запрос для получения проаудированной консолидированной финансовой отчётности Сбербанка

In [21]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 449/449 [00:06<00:00, 74.14question/s, accuracy=65.92%, correct=296, errors=0]


In [22]:
accuracy = result['correct'] / result['total']

conf_scores = []

correct_conf = []
incorrect_conf = []

for res in result['results']:
    confidence = predicted.get(res['question_id'], {}).get('confidence')
    if confidence is None:
        continue
    score = confidence if res['judge_score'] == 1 else 1 - confidence
    conf_scores.append(score)

    if res['judge_score'] == 1:
        correct_conf.append(confidence)
    else:
        incorrect_conf.append(confidence)

conf_precision = sum(conf_scores) / len(conf_scores) if conf_scores else float('nan')

print(f"Accuracy:                   {accuracy:.2%} ({result['correct']}/{result['total']})")
print(f"Confidence precision:       {conf_precision:.4f}")
print(f"Avg confidence (correct):   {sum(correct_conf)/len(correct_conf):.4f}")
print(f"Avg confidence (incorrect): {sum(incorrect_conf)/len(incorrect_conf):.4f}")

Accuracy:                   65.92% (296/449)
Confidence precision:       0.6853
Avg confidence (correct):   0.8466
Avg confidence (incorrect): 0.8227


In [23]:
for res in result['results'][0:5]:
    print(res['question'])
    print(res['gold_answer'])
    print(res['predicted_answer'])
    print(res['judge_reasoning'])
    print(res['judge_score'])
    print(res['question_id'])
    print('-' * 75)

Каковы чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года?
5 592 млн руб.
Чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года, составили 5 592 (единицы измерения не указаны в предоставленном фрагменте, но обычно это млн руб.).
Оба ответа указывают на одинаковую сумму чистых комиссионных доходов.
1
q_009c97884b8dc010
---------------------------------------------------------------------------
Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?
1,151 тыс. белорусских рублей

Предсказанный ответ пуст, поэтому он не соответствует золотому ответу.
0
q_00d660efcf3e4607
---------------------------------------------------------------------------
Какие макроэкономические факторы используются ВТБ в качестве базового набора для учёта макроэкономических ожиданий при расчёте PD в течение 12 месяцев согласно отчётнос